# Movies Project Starter

Use this starter notebook for the BAN 6003 final project option: **The Movies Dataset / MovieLens ratings**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset mixes movie-level tables and rating-event-level data. The key skill is controlling granularity.

- A common final ABT is **one row per movie**.
- Do **not** directly merge raw `ratings` into `movies` and assume the result is still one row per movie.
- Aggregate ratings to `movieId` first, then merge the rating summary into the movie table.
- Use `movieId` for ratings and `tmdbId` for curated credits/keywords.
- Average rating is unstable for movies with very few ratings. Use a minimum rating-count threshold before modeling.
- Be careful about leakage: post-release fields such as revenue, popularity, vote count, or rating count may not be valid if your question is pre-release prediction.

## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [2]:
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [3]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


movies: 9,082 rows x 21 columns


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995



ratings: 99,810 rows x 5 columns


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02



links: 9,125 rows x 3 columns


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0



credits: 9,082 rows x 3 columns


,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret



keywords: 9,082 rows x 2 columns


,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men


In [4]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,movies,9082,21,0,72
1,ratings,99810,5,0,0
2,links,9125,3,0,13
3,credits,9082,3,0,114
4,keywords,9082,2,0,761


### Key and Granularity Audit

Use the expected key for each table before planning joins. `ratings` is the exception: its row meaning is a user-movie rating event, so check the combined `userId`-`movieId` key rather than expecting `movieId` alone to be unique. `links` is a crosswalk; because `movies` already contains both IDs, use it for validation unless your analysis needs one of its fields.


In [5]:
key_audit = pd.DataFrame([
    {"table": "movies", "expected_key": "movieId", "duplicate_keys": movies["movieId"].duplicated().sum(), "missing_key_values": movies["movieId"].isna().sum()},
    {"table": "movies", "expected_key": "tmdbId", "duplicate_keys": movies["tmdbId"].duplicated().sum(), "missing_key_values": movies["tmdbId"].isna().sum()},
    {"table": "ratings", "expected_key": "userId + movieId", "duplicate_keys": ratings[["userId", "movieId"]].duplicated().sum(), "missing_key_values": ratings[["userId", "movieId"]].isna().sum().sum()},
    {"table": "links", "expected_key": "movieId", "duplicate_keys": links["movieId"].duplicated().sum(), "missing_key_values": links["movieId"].isna().sum()},
    {"table": "credits", "expected_key": "tmdbId", "duplicate_keys": credits["tmdbId"].duplicated().sum(), "missing_key_values": credits["tmdbId"].isna().sum()},
    {"table": "keywords", "expected_key": "tmdbId", "duplicate_keys": keywords["tmdbId"].duplicated().sum(), "missing_key_values": keywords["tmdbId"].isna().sum()},
])

key_audit


,table,expected_key,duplicate_keys,missing_key_values
0,movies,movieId,0,0
1,movies,tmdbId,0,0
2,ratings,userId + movieId,0,0
3,links,movieId,0,0
4,credits,tmdbId,0,0
5,keywords,tmdbId,0,0


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Movies notes:

- `movies` is movie-level, while `ratings` is user-movie rating-event-level.
- `credits` and `keywords` are curated movie-level tables connected by `tmdbId`.
- Budget and revenue may contain zero or missing-like values; describe this before using them.
- Decide whether your project is about audience ratings, catalog strategy, revenue, popularity, or genre/content patterns.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 1 written workspace

Use this cell to draft the narrative required by the Canvas Milestone 1 page.

- Business or research question:
- Why this question matters:
- Tables/files used:
- Row meaning of each important table:
- Early data quality issues:
- Initial cleaning plan:
- Possible target/outcome:
- Ethics, privacy, or governance concerns:






# Milestone 1: Initial Data Profile and Basic Cleaning Plan

## Business Problem and Decision Context

Movie studios and streaming platforms must decide which types of movies are most likely to receive strong audience reception. Since production, licensing, and marketing resources are limited, decision-makers can benefit from identifying movie characteristics that are associated with higher user ratings.

The intended users of this analysis are movie studios, streaming platforms, or content strategy teams that evaluate movies for development, acquisition, or promotion.

Analytical question: Which movie-level characteristics are associated with a movie receiving a high average user rating?

A later modeling version of this question is: Can movie characteristics available at or near release be used to classify whether a movie will receive a high average user rating?

For this project, a potential definition of a highly rated movie is one with an average user rating of at least 3.75 and at least 20 ratings.


## Dataset Description and Source

The dataset was provided as part of the BAN 6003 Final Project Option 1: Movies and Audience Ratings. The course GitHub repository contains the project starter notebook and the curated data files used for the project.

Source:  
https://github.com/tianhaiz/ban6003-project-option-1-movies-template

The project uses five course-provided CSV files stored in the `data/` folder:

- `movies_metadata.csv`
- `ratings.csv`
- `links.csv`
- `credits_curated.csv`
- `keywords_curated.csv`

The dataset includes movie metadata, user ratings, movie identifier links, credits, and keywords.


## Raw Files, Tables, and Row Meaning

| File | Initial Rows | Initial Columns | What One Row Represents |
| `movies_metadata.csv` | 9,082 | 21 | One movie and its movie-level metadata |
| `ratings.csv` | 99,810 | 5 | One user's rating event for one movie |
| `links.csv` | 9,125 | 3 | One movie identifier crosswalk record |
| `credits_curated.csv` | 9,082 | 3 | One movie's curated director and top-cast information |
| `keywords_curated.csv` | 9,082 | 2 | One movie's curated keyword information |

Important fields include movie identifiers such as `movieId` and `tmdbId`, movie characteristics such as title, genre, release date, runtime, budget, and revenue, and rating information such as `userId` and `rating`. The credits and keywords tables provide additional movie-level descriptive information.

The tables do not all have the same unit of analysis. The movies, credits, and keywords tables contain movie-level information, while the ratings table contains individual user-movie rating events. Therefore, the ratings data will need to be aggregated to the movie level before it can be combined into a one-row-per-movie analytical dataset.

## Initial Row and Column Counts

The initial profile shows:

- `movies`: 9,082 rows and 21 columns
- `ratings`: 99,810 rows and 5 columns
- `links`: 9,125 rows and 3 columns
- `credits`: 9,082 rows and 3 columns
- `keywords`: 9,082 rows and 2 columns

The tables have different numbers of rows because they represent different types and levels of information. In particular, the ratings table is much larger because a movie can be rated by many different users.


## Data Types

The initial data types are generally appropriate, but several fields require additional attention.

The `release_date` field is currently stored as text and should be converted to a datetime value before date-based analysis.

Fields such as `movieId`, `tmdbId`, and `userId` represent identifiers rather than numerical measurements even though they are stored using numeric data types.

Several fields contain categorical or structured text information. These include genres, production companies, production countries, cast, director, and keywords. Some of these fields may require parsing or standardization before they can be used in later analysis.

The `tmdbId` field should also be checked across tables to make sure its data type is consistent before using it as a merge key.


## Missing Values

The initial profile identifies missing information in several tables.

The movie metadata contains some missing information in fields such as genre and status. The links table contains some missing `tmdbId` values. The credits data contains missing director and top-cast information.

The keywords table contains the largest amount of missing information, with some movies having no available keyword list.

The ratings table does not show the same level of missingness and appears relatively complete at the initial profiling stage.

These missing values will be investigated before deciding whether they should be retained, filled, categorized as unavailable, or excluded from specific analyses.


## Duplicate Records and Duplicate Keys

The initial profile found no exact duplicate rows in the five tables.

However, exact duplicate rows are different from duplicate keys, so the important identifier fields must also be checked separately.

Expected keys include:

- `movies`: `movieId` and `tmdbId`
- `ratings`: combination of `userId` and `movieId`
- `links`: `movieId`
- `credits`: `tmdbId`
- `keywords`: `tmdbId`

Repeated `movieId` values in the ratings table are expected because many users can rate the same movie. Therefore, the combination of `userId` and `movieId` is a more appropriate key for identifying potentially duplicated rating events.

The profiling code below performs these key-level duplicate checks.


## Suspicious, Impossible, or Confusing Values

Several variables require additional investigation before analysis.

The `budget` and `revenue` fields contain zero values. A zero could represent a legitimate value, but it could also indicate that financial information was unavailable or unreported. These values should therefore be investigated before treating them as actual financial amounts.

Runtime should be checked for zero, negative, or unusually large values.

Ratings should be checked to confirm that they remain within the expected rating range in the dataset.

Release dates should be converted to datetime values and checked for missing or invalid dates.

Variables such as popularity, vote average, and vote count should also be examined for unusual ranges or outliers.

Finally, the links table contains more rows than the movies table. The movie identifiers in these tables should therefore be compared to determine whether any records do not have a corresponding movie.

The profiling code below performs these initial suspicious-value and identifier checks.


## Initial Cleaning Plan

Based on the initial profiling, the planned cleaning process is:

1. Convert `release_date` into a proper datetime variable.
2. Verify and standardize identifier data types, especially `tmdbId`.
3. Check expected keys for duplicate values and investigate any unexpected duplicate keys.
4. Investigate missing genre, director, cast, keyword, and identifier information.
5. Investigate zero `budget` and `revenue` values to determine whether they represent actual zeros or unavailable information.
6. Check numerical variables such as runtime, ratings, budget, revenue, popularity, and vote information for impossible or suspicious values.
7. Standardize or parse categorical and text variables when necessary.
8. Aggregate the ratings table to one row per `movieId` before combining it with movie-level information.
9. Merge the datasets using the appropriate identifiers, such as `movieId` and `tmdbId`.
10. Validate row counts, missing values, and duplicate keys after major transformations.

These cleaning decisions will be updated as additional profiling and transformation work is completed.


## Potential Target or Outcome Variable

A potential target variable for later analysis is `high_user_rating`.

This could be created as a binary variable indicating whether a movie has an average user rating of at least 3.75. To reduce the effect of movies with very few ratings, a movie would also need at least 20 ratings before being assigned the target.

The continuous average user rating, such as `user_rating_mean`, could also be considered as an alternative outcome if regression is more appropriate later in the project.

The final target and modeling method will be confirmed after the ratings are aggregated and the distribution and coverage of the proposed outcome are examined.

Care will also be taken to avoid data leakage. If the goal is to predict audience reception using information available near a movie's release, information that only becomes available after release should not be used as a predictor.


## Ethics, Privacy, Governance, and Responsible Use

The ratings data uses numeric user identifiers rather than direct personal identifiers. However, user-level behavioral information should still be handled responsibly and only used for the purposes of this project.

The users represented in the dataset may not represent the preferences of all movie audiences. Therefore, the results should not automatically be generalized to every audience or population.

Movie ratings may also contain selection bias because users choose which movies they watch and rate. More visible or popular movies may receive substantially more ratings than less visible movies.

The analysis should not be interpreted as measuring objective artistic quality or proving that a particular movie characteristic causes higher ratings. Relationships found in the data should be interpreted as associations unless the research design supports stronger conclusions.

If people-related variables such as director or cast are used later, the results should not be treated as objective judgments about those individuals. The analysis is intended to support business decision-making rather than replace human judgment.


## Reproducibility

The project can be reproduced using the course-provided GitHub repository:

https://github.com/tianhaiz/ban6003-project-option-1-movies-template

The five required CSV files are stored in the project's `data/` folder. The notebook should be run from the setup and data-loading sections through the Milestone 1 profiling section.

Running the notebook from top to bottom allows the row counts, data types, missing-value checks, duplicate checks, and suspicious-value checks to be reproduced from the original project data.

In [9]:
# Milestone 1 code workspace
# Add your additional profiling checks here.
# Examples: missingness by important columns, duplicate key checks, suspicious value checks.



# 1) Table-level profile
profile = pd.DataFrame([
    {
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
    }
    for name, df in tables.items()
])
display(profile)

# 2) Missing values by column
for name, df in tables.items():
    missing = (
        pd.DataFrame({
            "missing_count": df.isna().sum(),
            "missing_pct": (df.isna().mean() * 100).round(2),
        })
        .query("missing_count > 0")
        .sort_values(["missing_count", "missing_pct"], ascending=False)
    )
    print(f"\nMissing values: {name}")
    display(missing if not missing.empty else pd.DataFrame({"result": ["No missing values"]}))

# 3) Expected-key audit
key_audit = pd.DataFrame([
    {
        "table": "movies",
        "expected_key": "movieId",
        "duplicate_keys": int(movies["movieId"].duplicated().sum()),
        "missing_key_values": int(movies["movieId"].isna().sum()),
    },
    {
        "table": "movies",
        "expected_key": "tmdbId",
        "duplicate_keys": int(movies["tmdbId"].duplicated().sum()),
        "missing_key_values": int(movies["tmdbId"].isna().sum()),
    },
    {
        "table": "ratings",
        "expected_key": "userId + movieId",
        "duplicate_keys": int(ratings[["userId", "movieId"]].duplicated().sum()),
        "missing_key_values": int(ratings[["userId", "movieId"]].isna().sum().sum()),
    },
    {
        "table": "links",
        "expected_key": "movieId",
        "duplicate_keys": int(links["movieId"].duplicated().sum()),
        "missing_key_values": int(links["movieId"].isna().sum()),
    },
    {
        "table": "credits",
        "expected_key": "tmdbId",
        "duplicate_keys": int(credits["tmdbId"].duplicated().sum()),
        "missing_key_values": int(credits["tmdbId"].isna().sum()),
    },
    {
        "table": "keywords",
        "expected_key": "tmdbId",
        "duplicate_keys": int(keywords["tmdbId"].duplicated().sum()),
        "missing_key_values": int(keywords["tmdbId"].isna().sum()),
    },
])
display(key_audit)

# 4) Data types
for name, df in tables.items():
    print(f"\nData types: {name}")
    display(df.dtypes.astype(str).to_frame("dtype"))

# 5) Suspicious/confusing-value checks
release_date_parsed = pd.to_datetime(movies["release_date"], errors="coerce")

suspicious_checks = pd.Series({
    "movies_budget_zero_or_negative": int((pd.to_numeric(movies["budget"], errors="coerce") <= 0).sum()),
    "movies_revenue_zero_or_negative": int((pd.to_numeric(movies["revenue"], errors="coerce") <= 0).sum()),
    "movies_runtime_zero_or_negative": int((pd.to_numeric(movies["runtime"], errors="coerce") <= 0).sum()),
    "movies_runtime_over_300_minutes": int((pd.to_numeric(movies["runtime"], errors="coerce") > 300).sum()),
    "movies_unparseable_or_missing_release_date": int(release_date_parsed.isna().sum()),
    "ratings_below_0_5_or_above_5": int(((ratings["rating"] < 0.5) | (ratings["rating"] > 5)).sum()),
    "links_missing_tmdbId": int(links["tmdbId"].isna().sum()),
    "movies_missing_primary_genre": int(movies["primary_genre"].isna().sum()),
    "credits_missing_director": int(credits["director"].isna().sum()),
    "credits_missing_top_cast": int(credits["top_cast"].isna().sum()),
    "keywords_missing_keywords_list": int(keywords["keywords_list"].isna().sum()),
})
display(suspicious_checks.to_frame("count"))

# 6) Useful category/range checks for interpretation
print("\nRating distribution:")
display(ratings["rating"].value_counts().sort_index().to_frame("count"))

print("\nMovie status distribution:")
display(movies["status"].fillna("<MISSING>").value_counts().to_frame("count"))

print("\nNumeric ranges for selected movie fields:")
numeric_fields = ["runtime", "budget", "revenue", "popularity", "vote_average", "vote_count", "release_year"]
display(movies[numeric_fields].describe().T)

# 7) Crosswalk coverage check
movie_ids = set(movies["movieId"].dropna())
link_ids = set(links["movieId"].dropna())
print(f"movieIds in links but not movies: {len(link_ids - movie_ids):,}")
print(f"movieIds in movies but not links: {len(movie_ids - link_ids):,}")



,table,rows,columns,duplicate_rows,missing_cells
0,movies,9082,21,0,72
1,ratings,99810,5,0,0
2,links,9125,3,0,13
3,credits,9082,3,0,114
4,keywords,9082,2,0,761



Missing values: movies


,missing_count,missing_pct
genres_list,35,0.39
primary_genre,35,0.39
status,2,0.02



Missing values: ratings


,result
0,No missing values


...


Movie status distribution:


,count
status,
Released,9061
Rumored,11
Post Production,7
<MISSING>,2
In Production,1



Numeric ranges for selected movie fields:


,count,mean,std,min,25%,50%,75%,max
runtime,9082.0,1.056614e+02,3.035071e+01,0.000000,93.000000,102.000000,1.150000e+02,1.140000e+03
budget,9082.0,1.664789e+07,3.345276e+07,0.000000,0.000000,100000.000000,2.000000e+07,3.800000e+08
revenue,9082.0,4.918728e+07,1.301801e+08,0.000000,0.000000,271608.000000,3.672528e+07,2.787965e+09
popularity,9082.0,7.268905e+00,9.372314e+00,0.000004,2.954521,6.418871,9.861742e+00,5.474883e+02
vote_average,9082.0,6.362178e+00,1.038748e+00,0.000000,5.800000,6.500000,7.000000e+00,1.000000e+01
vote_count,9082.0,4.390509e+02,9.961565e+02,0.000000,29.000000,96.000000,3.657500e+02,1.407500e+04
release_year,9082.0,1.991952e+03,1.938771e+01,1902.000000,1984.000000,1997.000000,2.006000e+03,2.016000e+03


movieIds in links but not movies: 43
movieIds in movies but not links: 0


## Interpretation of Initial Profiling Results

#The initial profiling shows that the five datasets are generally well structured, but several data quality issues should be addressed before integration and modeling. No exact duplicate rows were found in any of the five tables. The expected key checks also found zero duplicate keys and zero missing values in the primary key fields that were tested. This suggests that the datasets have a strong initial structure for later integration.



### Missing Values

#Missing values are concentrated in a relatively small number of variables. The movie metadata contains 35 missing `genres_list` values, 35 missing `primary_genre` values, and 2 missing `status` values. The ratings table contains no missing values.

#The links table contains 13 missing `tmdbId` values. The credits table contains 24 missing directors and 90 missing top-cast values. The largest missingness issue occurs in the keywords table, where 761 movies, or 8.38% of the records, are missing `keywords_list`.

#These missing values will need to be handled based on how each variable is used. Missing descriptive information such as keywords or cast should not automatically cause an entire movie record to be removed.




### Data Types

#Most numerical fields have appropriate numeric data types, while categorical and text variables are stored as strings. The `release_date` variable is currently stored as a string and should be converted to a datetime variable before date-based analysis. In contrast, `rating_datetime` in the ratings table is already stored as a datetime.

#The `tmdbId` variable is stored as an integer in the movies, credits, and keywords tables but as a float in the links table because that table contains 13 missing `tmdbId` values. This difference will need to be addressed before using `tmdbId` as an integration key.




### Suspicious and Confusing Values

#The profiling identified several values that require additional investigation. There are 4,443 movies with budget values less than or equal to zero and 4,322 movies with revenue values less than or equal to zero. Because such a large portion of the dataset contains these values, zeros may represent unavailable or unreported financial information rather than actual zero-dollar budgets or revenues.

#Runtime also contains potentially suspicious observations. There are 20 movies with runtimes less than or equal to zero and 15 movies with runtimes greater than 300 minutes. The maximum runtime is 1,140 minutes, so these extreme observations should be investigated before deciding whether they are valid or should be treated as data quality issues.

#No ratings were found outside the expected 0.5 to 5.0 range, and no release dates were missing or unable to be parsed. This indicates that these fields are relatively consistent.




### Rating and Movie Status Findings

#The ratings use half-point increments from 0.5 through 5.0. The most common individual rating is 4.0, with 28,705 rating events, followed by 3.0 with 20,039 and 5.0 with 15,043.

#Most movies have a status of `Released`, with 9,061 of the 9,082 movie records in this category. A small number are categorized as `Rumored`, `Post Production`, or `In Production`, and two records have missing status values. These uncommon status categories should be considered when deciding which movies are appropriate for the final analysis.




### Table Relationships and Integration

#The crosswalk comparison found 43 `movieId` values in the links table that do not appear in the movies table. In contrast, there are zero `movieId` values in the movies table that are missing from the links table.

#This explains the difference between the 9,125 rows in the links table and the 9,082 rows in the movies table. The additional link records should not automatically be included in the final analytical dataset because they do not have corresponding movie metadata.

#Another important structural issue is the difference in granularity between tables. The ratings table contains 99,810 individual user-movie rating events, while the main movie table contains one row per movie. Therefore, ratings must be aggregated by `movieId` before they can be merged into a one-row-per-movie analytical base table.




### Next Steps

#Based on these profiling results, the next stage will focus on converting `release_date` to a datetime, standardizing identifier data types, investigating zero budget and revenue values, reviewing suspicious runtime values, and developing appropriate approaches for missing genre, credit, and keyword information.

#The ratings data will later be aggregated to the movie level before integration. Row counts, missingness, and duplicate keys will be checked again after major cleaning, transformation, and integration steps to confirm that the analytical dataset maintains the intended one-row-per-movie structure.





In [7]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


movies


,dtype
movieId,int64
tmdbId,int64
imdb_id,str
title,str
original_title,str
original_language,str
release_date,str
runtime,float64
budget,int64
revenue,float64


,missing_rate
genres_list,0.003854
primary_genre,0.003854
status,0.000220
tmdbId,0.000000
movieId,0.000000
original_title,0.000000
title,0.000000
imdb_id,0.000000
original_language,0.000000
revenue,0.000000



ratings


,dtype
userId,int64
movieId,int64
rating,float64
timestamp,int64
rating_datetime,datetime64[us]


...

,dtype
tmdbId,int64
director,str
top_cast,str


,missing_rate
top_cast,0.009910
director,0.002643
tmdbId,0.000000



keywords


,dtype
tmdbId,int64
keywords_list,str


,missing_rate
keywords_list,0.083792
tmdbId,0.000000


## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what keys define one row, and what tables were joined or aggregated.

Critical integration reminder:

Before creating a movie-level ABT, aggregate ratings. Examples include:

- average rating by `movieId`
- rating count by `movieId`
- rating standard deviation by `movieId`
- first and last rating date by `movieId`
- distinct user count by `movieId`

After merging, validate that `movieId` is not duplicated in the ABT.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

- Final or preliminary unit of analysis:
- Primary key for one row:
- Tables aggregated before joining:
- Tables joined directly:
- Row-count validation after each major merge:
- New variables created:
- Remaining data quality concerns:



In [ ]:
# Milestone 2 code workspace
# Build or revise your ABT here.
# Add row-count checks and duplicate-key checks after each major merge.



In [6]:
rating_summary = (
    ratings.groupby("movieId")
    .agg(
        user_rating_mean=("rating", "mean"),
        user_rating_count=("rating", "size"),
        user_rating_std=("rating", "std"),
        first_rating_date=("rating_datetime", "min"),
        last_rating_date=("rating_datetime", "max")
    )
    .reset_index()
)

abt = (
    movies
    .merge(rating_summary, on="movieId", how="left", validate="one_to_one")
    .merge(credits, on="tmdbId", how="left", validate="one_to_one")
    .merge(keywords, on="tmdbId", how="left", validate="one_to_one")
)

abt["has_revenue"] = (pd.to_numeric(abt["revenue"], errors="coerce") > 0).astype("Int64")
abt["rating_target_eligible"] = abt["user_rating_count"].ge(20)
abt["high_user_rating"] = (
    abt["user_rating_mean"].ge(3.75)
    .where(abt["rating_target_eligible"])
    .astype("Int64")
)


In [7]:
abt_validation = pd.Series({
    "movie_rows_before_merges": len(movies),
    "abt_rows_after_merges": len(abt),
    "duplicate_movie_ids": abt["movieId"].duplicated().sum(),
    "duplicate_tmdb_ids": abt["tmdbId"].duplicated().sum(),
    "movies_with_rating_summary": abt["user_rating_count"].notna().sum(),
    "movies_eligible_for_rating_target": abt["high_user_rating"].notna().sum(),
})

display(abt_validation)
display(abt.head())
display(abt.isna().mean().sort_values(ascending=False).head(15).to_frame("missing_rate"))


Preliminary ABT shape: (9082, 32)


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year,user_rating_mean,user_rating_count,user_rating_std,first_rating_date,last_rating_date,director,top_cast,keywords_list,has_revenue,high_user_rating,well_rated_enough_data
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995,3.872470,247.0,0.958981,1996-03-30 19:00:13,2016-10-06 19:55:11,John Lasseter,Tom Hanks|Tim Allen|Don Rickles,jealousy|toy|boy|friendship|friends|rivalry|bo...,1,1,1
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995,3.401869,107.0,0.880714,1996-03-30 19:12:30,2016-08-01 17:42:33,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst,board game|disappearance|based on children's b...,1,0,0
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995,3.161017,59.0,1.150115,1996-06-05 06:19:04,2016-08-16 22:07:21,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret,fishing|best friend|duringcreditsstinger|old men,0,0,0
3,4,31357,tt0114885,Waiting to Exhale,Waiting to Exhale,en,1995-12-22,127.0,16000000,81452156.0,3.859495,6.1,34.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Drama|Romance,Comedy,1995,2.384615,13.0,0.938835,1996-06-10 16:45:35,2004-07-27 06:14:12,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine,based on novel|interracial relationship|single...,1,0,0
4,5,11862,tt0113041,Father of the Bride Part II,Father of the Bride Part II,en,1995-02-10,106.0,0,76578911.0,8.387519,5.7,173.0,"[{'id': 35, 'name': 'Comedy'}]","[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy,Comedy,1995,3.267857,56.0,0.948512,1996-04-14 14:23:59,2016-08-16 22:15:47,Charles Shyer,Steve Martin|Diane Keaton|Martin Short,baby|midlife crisis|confidence|aging|daughter|...,1,0,0


,missing_rate
user_rating_std,0.341224
keywords_list,0.083792
top_cast,0.009910
last_rating_date,0.006276
first_rating_date,0.006276
user_rating_count,0.006276
user_rating_mean,0.006276
primary_genre,0.003854
genres_list,0.003854
director,0.002643


### SQL Validation Connected to the ABT

The project milestone asks for database or SQL work that supports the workflow. This in-memory SQLite check verifies the movie-level ABT row count and key uniqueness without creating another permanent database file.


In [ ]:
import sqlite3

with sqlite3.connect(":memory:") as connection:
    abt.to_sql("movies_abt", connection, if_exists="replace", index=False)
    sql_abt_check = pd.read_sql_query(
        """
        SELECT
            COUNT(*) AS abt_rows,
            COUNT(DISTINCT movieId) AS unique_movie_ids,
            SUM(CASE WHEN high_user_rating IS NULL THEN 1 ELSE 0 END) AS target_not_defined_rows
        FROM movies_abt;
        """,
        connection,
    )

sql_abt_check


### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [ ]:
data_dictionary = pd.DataFrame({
    "column_name": abt.columns,
    "data_type": [str(abt[column].dtype) for column in abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": "",
})

# Hint: use .loc[] to document one column at a time, following the Week 11 pattern.
data_dictionary.head()


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminder:

- If you model `high_user_rating`, use only rows where that nullable target is defined; these movies have at least 20 ratings.
- State when the prediction is meant to occur. For a pre-release question, exclude post-release fields such as revenue, popularity, vote count, rating summaries, and rating dates.
- Do not interpret a rating model as proving artistic quality or causal effects.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [ ]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


In [8]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
movieId,9082.0,NaN,NaN,NaN,30963.741577,1.0,2843.25,6265.5,56026.25,163949.0,40659.690679
tmdbId,9082.0,NaN,NaN,NaN,38710.463885,2.0,9446.25,15775.0,39010.25,416437.0,62123.411079
imdb_id,9082,9082,tt0114709,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,9082,8809,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_title,9082,8841,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_language,9082,42,en,7947,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_date,9082,6002,1994-01-01,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
runtime,9082.0,NaN,NaN,NaN,105.661418,0.0,93.0,102.0,115.0,1140.0,30.350712
budget,9082.0,NaN,NaN,NaN,16647890.737833,0.0,0.0,100000.0,20000000.0,380000000.0,33452764.781378
revenue,9082.0,NaN,NaN,NaN,49187276.79586,0.0,0.0,271608.0,36725282.5,2787965087.0,130180082.298936


## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [ ]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "movies_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "movies_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "movies_model_metrics.csv"
# report_path = REPORTS_DIR / "movies_final_report.html"


In [9]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "movies_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "movies_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
